# 😨 Sovereign-Dispatch — Model 3: Panic Detection

**Tujuan:** Skor kepanikan 0.0–1.0 dari pesan darurat  
**Arsitektur:** Rule-based hybrid + Optional fine-tuned IndoBERT  
**Output:** `PanicResult` dengan score, level, sinyal aktif, dispatch flag

---
### Level Kepanikan
| Skor | Level | Aksi |
|------|-------|------|
| 0.00–0.19 | TENANG | Log saja |
| 0.20–0.39 | WASPADA | Antri biasa |
| 0.40–0.64 | PANIK | Prioritas tinggi |
| 0.65–1.00 | KRITIS | Dispatch otomatis |

---
### Struktur Notebook
1. Install & Import
2. Rule-based Engine
3. Test Rule-based
4. Zero-shot ML (tanpa training)
5. Siapkan Dataset Fine-tune
6. Fine-tune IndoBERT
7. Evaluasi & Confusion Matrix
8. Ensemble Rule + ML
9. Simpan & Inferensi

## 1. Install & Import

In [ ]:
!pip install transformers datasets seqeval scikit-learn seaborn matplotlib torch -q

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Optional
from sklearn.metrics import confusion_matrix, classification_report, f1_score

print('Import selesai')

## 2. Rule-based Engine

In [ ]:
# Import dari file modul
# Pastikan panic_detector.py ada di folder yang sama
from panic_detector import analyze, batch_analyze, PanicResult, DISPATCH_THRESHOLD

print('panic_detector loaded OK')

## 3. Test Rule-based

In [ ]:
test_messages = [
    'ada orang jatuh, tolong bantu ya',
    'Tlg! ibu luka di Gampong Tibang segera!',
    'TOLONG!! bapak sy patah tulang di Jalan Diponegoro sekarang!!',
    'DARURAT!! tlg tolong ibu sy di Gampong Lambhuk luka parah SEGERA!!!',
    'aneuk hana sadar lam Gampong Pande, tulong!!!',
    'Siti Rahma pingsan dekat Pasar Aceh jinoe!!!',
    'tlg sy butuh bantuan medis segera di Desa Kajhu',
    'SOS!!! Muhammad Iqbal terkubur di reruntuhan Gampong Lambhuk!!!',
]

results = [analyze(m) for m in test_messages]

for r in sorted(results, key=lambda x: x.score, reverse=True):
    print(r.summary())
    print()

In [ ]:
# Visualisasi skor
fig, ax = plt.subplots(figsize=(10, 4))
labels_short = [m[:35]+'...' if len(m)>35 else m for m in test_messages]
scores = [analyze(m).score for m in test_messages]
colors = ['#639922' if s<0.2 else '#BA7517' if s<0.4 else '#D85A30' if s<0.65 else '#E24B4A' for s in scores]

bars = ax.barh(labels_short, scores, color=colors)
ax.axvline(x=DISPATCH_THRESHOLD, color='#E24B4A', linestyle='--', linewidth=1.2, label=f'Dispatch threshold ({DISPATCH_THRESHOLD})')
ax.set_xlabel('Panic Score')
ax.set_xlim(0, 1.0)
ax.set_title('Panic Score per Pesan')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Zero-shot ML (Tanpa Training — MVP Cepat)

In [ ]:
# Gunakan mode ini dulu sebelum ada data berlabel
# Model: joeddav/xlm-roberta-large-xnli (multilingual)

from transformers import pipeline

zshot = pipeline(
    'zero-shot-classification',
    model='joeddav/xlm-roberta-large-xnli',
)

# Test satu pesan
msg = 'DARURAT!! tlg tolong ibu sy di Gampong Lambhuk luka parah SEGERA!!!'
result = zshot(msg, candidate_labels=['darurat mendesak', 'pesan biasa'])
print(result)

In [ ]:
# Ensemble rule-based + zero-shot
for msg in test_messages[:4]:
    r = analyze(msg, ml_mode='zshot', ml_weight=0.4)
    print(f'[{r.level}] rule={r.rule_score:.2f} ml={r.ml_score:.2f} final={r.score:.2f}')
    print(f'  {msg[:70]}')
    print()

## 5. Siapkan Dataset Fine-tune

In [ ]:
# Buat dataset berlabel dari rule-based (bootstrap)
# Gunakan ini sebagai titik awal, lalu review manual

import json

# Load ner_dataset_v2.json sebagai sumber teks
with open('ner_dataset_v2.json') as f:
    ner_data = json.load(f)

# Rekonstruksi kalimat dari token
sentences = [' '.join(entry['tokens']) for entry in ner_data]

# Auto-label pakai rule-based, threshold 0.4 = panik
labeled = []
for text in sentences:
    r = analyze(text)
    labeled.append({
        'text':  text,
        'label': 1 if r.score >= 0.40 else 0,
        'score': r.score
    })

df = pd.DataFrame(labeled)
print(df['label'].value_counts())
print(f'\nTotal: {len(df)} samples')
df.head(10)

In [ ]:
df[['text','label']].to_csv('panic_dataset.csv', index=False)
print('Disimpan ke panic_dataset.csv')
print('PENTING: Review manual label sebelum fine-tune!')

## 6. Fine-tune IndoBERT

In [ ]:
from panic_detector import prepare_finetune_dataset, finetune

# Siapkan dataset HuggingFace
ds = prepare_finetune_dataset(
    labeled_csv='panic_dataset.csv',
    output_dir='./panic_hf_dataset',
    test_size=0.2
)

print(ds)

In [ ]:
# Training ~5 menit di GPU Colab T4
trainer = finetune(
    dataset_dir   = './panic_hf_dataset',
    output_dir    = './panic_model',
    model_name    = 'indobenchmark/indobert-base-p1',
    epochs        = 5,
    batch_size    = 16,
    learning_rate = 2e-5,
)
print('Fine-tune selesai!')

## 7. Evaluasi & Confusion Matrix

In [ ]:
# Ambil prediksi dari trainer
from datasets import load_from_disk

ds = load_from_disk('./panic_hf_dataset')
preds_output = trainer.predict(ds['test'])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print(classification_report(y_true, y_pred, target_names=['Tidak Panik','Panik']))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Panik','Panik'],
            yticklabels=['Tidak Panik','Panik'], ax=ax)
ax.set_xlabel('Prediksi')
ax.set_ylabel('Aktual')
ax.set_title('Confusion Matrix — Panic Detector')
plt.tight_layout()
plt.show()

## 8. Ensemble Rule + Fine-tuned

In [ ]:
from panic_detector import load_finetuned_model

load_finetuned_model('./panic_model')

for msg in test_messages:
    r = analyze(msg, ml_mode='finetuned', ml_weight=0.4)
    dispatch_mark = ' ← DISPATCH' if r.dispatch else ''
    print(f'[{r.level:<8}] {r.score:.3f}  {msg[:60]}{dispatch_mark}')

## 9. Simpan & Inferensi

In [ ]:
# Contoh integrasi ke pipeline Sovereign-Dispatch

def process_incoming_message(raw_text: str) -> dict:
    """Entry point dari sistem dispatch."""
    result = analyze(raw_text, ml_mode='none')  # ganti 'finetuned' setelah training
    return {
        'original_text': raw_text,
        'panic_score':   result.score,
        'panic_level':   result.level,
        'auto_dispatch': result.dispatch,
        'top_signals':   [(s.name, round(s.score,2)) for s in result.signals if s.score > 0],
    }

# Test
out = process_incoming_message('DARURAT!! ibu sy tidak sadar di Gampong Lambhuk SEGERA!!!')
for k, v in out.items():
    print(f'{k:20}: {v}')